# 01 — Image Collection

Pulls 100-200k diverse images from open sources into Cloudflare R2 and writes a Parquet manifest.

**Runs entirely in Colab.** Your laptop is not used. Resumable across session timeouts.

Before running:
1. Clone the repo into Colab (or mount Drive)
2. Upload `.env` with your keys (R2, HF, Unsplash, Pexels) to the notebook's runtime
3. Configure caps and sources in `config/config.yaml` as needed

In [ ]:
# Clone repo into Colab (run once per session)
import os, pathlib
if not pathlib.Path('zahava-tzniut').exists():
    !git clone https://github.com/YOUR_USERNAME/zahava-tzniut.git
os.chdir('zahava-tzniut')
!pwd && ls

In [ ]:
# Install deps
!pip install -q -r requirements.txt
!pip install -q fiftyone

In [ ]:
# Upload .env to /content/zahava-tzniut/.env via the Files panel before running this cell
from pathlib import Path
assert Path('.env').exists(), 'upload .env first'
from pipelines.common import load_config
cfg = load_config()
print(f"target_total: {cfg['collection']['target_total']}")
print(f"per_source_caps: {cfg['collection']['per_source_caps']}")

In [ ]:
# Run all collectors. Resumable — re-run this cell if Colab times out.
from pipelines.collection.run_all import run
run()  # to scope: run(only=['unsplash','pexels'])

In [ ]:
# Inspect the deduped manifest
import pandas as pd
df = pd.read_parquet('manifests/collection_deduped.parquet')
print(f"total images: {len(df)}")
print('\nper-source counts:')
print(df['source'].value_counts())
print('\nfile size stats (KB):')
print((df['file_size']/1024).describe())

In [ ]:
# Push manifest to HF Datasets (so labeling notebook + training notebook can read it)
from huggingface_hub import HfApi
from pipelines.common import require_env
api = HfApi(token=require_env('HF_TOKEN'))
repo = require_env('HF_DATASET_REPO')
api.create_repo(repo, repo_type='dataset', exist_ok=True)
api.upload_file(path_or_fileobj='manifests/collection_deduped.parquet',
                path_in_repo='collection_deduped.parquet',
                repo_id=repo, repo_type='dataset')
print(f'manifest pushed to {repo}')